# Privacy and Data Protection
## Module 5, Lesson 4

In this notebook, we will:
1. Implement the Laplace mechanism for differential privacy
2. Analyze the privacy-utility trade-off
3. Simulate a membership inference attack
4. Implement k-anonymity

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(42)

## 1. Differential Privacy: Laplace Mechanism

We implement the Laplace mechanism for releasing summary statistics with formal privacy guarantees.

In [ ]:
def laplace_mechanism(true_value, sensitivity, epsilon):
    """Add Laplace noise to achieve epsilon-differential privacy."""
    noise = np.random.laplace(0, sensitivity / epsilon)
    return true_value + noise

# Generate synthetic salary data (USD)
n = 10000
salaries = np.random.lognormal(mean=11.0, sigma=0.5, size=n)
true_mean = np.mean(salaries)
true_median = np.median(salaries)

print(f"True mean: ${true_mean:.2f}")
print(f"True median: ${true_median:.2f}")

In [ ]:
epsilon_values = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
n_trials = 1000

results = []
for eps in epsilon_values:
    errors_mean = []
    errors_median = []
    for _ in range(n_trials):
        # Sensitivity of mean: range / n
        data_range = salaries.max() - salaries.min()
        private_mean = laplace_mechanism(true_mean, data_range / n, eps)
        errors_mean.append(abs(private_mean - true_mean))
        
        # Sensitivity of median: ~1 / (n * f(median))
        # For simplicity, use a common approximation
        private_median = laplace_mechanism(true_median, 2.0 / n, eps)
        errors_median.append(abs(private_median - true_median))
    
    results.append({
        'epsilon': eps,
        'mae_mean': np.mean(errors_mean),
        'std_mean': np.std(errors_mean),
        'mae_median': np.mean(errors_median),
        'std_median': np.std(errors_median)
    })

results_df = pd.DataFrame(results)
print(results_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE vs Epsilon
axes[0].plot(results_df['epsilon'], results_df['mae_mean'], 'o-', label='Mean')
axes[0].plot(results_df['epsilon'], results_df['mae_median'], 's-', label='Median')
axes[0].set_xscale('log')
axes[0].set_xlabel('Epsilon (privacy budget)')
axes[0].set_ylabel('Mean Absolute Error')
axes[0].set_title('Privacy-Utility Trade-off')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution for a specific epsilon
eps_to_plot = 1.0
private_values = [laplace_mechanism(true_mean, (salaries.max() - salaries.min()) / n, eps_to_plot) 
                  for _ in range(5000)]
axes[1].hist(private_values, bins=50, alpha=0.7, label=f'epsilon={eps_to_plot}')
axes[1].axvline(true_mean, color='r', linestyle='--', linewidth=2, label='True mean')
axes[1].set_xlabel('Private Mean')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Distribution of Private Means (epsilon={eps_to_plot})')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Membership Inference Attack

We simulate a membership inference attack: an attacker tries to determine whether a specific record was in the training set.

In [ ]:
def simulate_membership_inference(n_train=500, n_holdout=500, n_features=10):
    # Generate data
    X = np.random.randn(n_train + n_holdout, n_features)
    y = (X[:, 0] + X[:, 1] > 0).astype(int)
    
    X_train = X[:n_train]
    y_train = y[:n_train]
    X_holdout = X[n_train:]
    y_holdout = y[n_train:]
    
    # Train model
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    # Get confidence scores
    train_conf = model.predict_proba(X_train).max(axis=1)
    holdout_conf = model.predict_proba(X_holdout).max(axis=1)
    
    return train_conf, holdout_conf

train_conf, holdout_conf = simulate_membership_inference()

plt.figure(figsize=(10, 4))
plt.hist(train_conf, bins=30, alpha=0.7, label='Training set (member)')
plt.hist(holdout_conf, bins=30, alpha=0.7, label='Holdout set (non-member)')
plt.xlabel('Model confidence')
plt.ylabel('Frequency')
plt.title('Membership Inference: Confidence Distribution')
plt.legend()
plt.show()

print(f"Mean confidence (member): {train_conf.mean():.3f}")
print(f"Mean confidence (non-member): {holdout_conf.mean():.3f}")
print("\nIf the distributions differ significantly, an attacker can infer membership.")

## 3. Simple k-Anonymity

We demonstrate k-anonymity by generalizing quasi-identifiers.

In [ ]:
data = pd.DataFrame({
    'age': [25, 27, 45, 48, 52, 31, 29, 55, 58, 33],
    'zip': ['12345', '12345', '12346', '12346', '12347', '12345', '12345', '12346', '12346', '12345'],
    'diagnosis': ['A', 'B', 'A', 'C', 'B', 'A', 'B', 'C', 'A', 'B']
})

print("Original data:")
print(data)
print("\nUnique combinations of (age, zip):", data.groupby(['age', 'zip']).ngroups)
print("Each record is unique!")

In [ ]:
# Generalize age to 10-year bands to achieve 2-anonymity
data_k2 = data.copy()
data_k2['age'] = pd.cut(data_k2['age'], bins=[20, 30, 40, 50, 60], labels=['20-29', '30-39', '40-49', '50-59'])

print("After generalization (age bands):")
print(data_k2)
group_sizes = data_k2.groupby(['age', 'zip']).size()
print(f"\nMinimum group size: {group_sizes.min()}")
print(f"Achieves 2-anonymity: {group_sizes.min() >= 2}")

## 4. Exercises

### Basic
1. What happens to the MAE when epsilon is very small (0.01) vs. very large (10)? Why?
2. In the membership inference simulation, what would happen if the model overfits heavily?

### Implementation
3. Implement the Gaussian mechanism (add Gaussian noise) instead of Laplace. Compare the noise distributions.
4. Create a function that computes the privacy budget consumed by releasing k statistics under composition (epsilon_total = k * epsilon).

### Critical Thinking
5. The membership inference attack shown here is very simple. What additional information could an attacker use to improve their inference? How would you defend against more sophisticated attacks?
6. k-anonymity has known weaknesses. Research "homogeneity attack" and "background knowledge attack." Can you construct an example where 2-anonymity fails?
7. A hospital wants to release a differentially private version of their model for research. What epsilon should they choose? How would they determine the right privacy-utility trade-off?